# 01 — Baseline GPT-style Causal LM Training

**Goal:** Train a ~20 M-parameter GPT-style decoder-only transformer on 50 M tokens from the [TinyStories](https://huggingface.co/datasets/roneneldan/TinyStories) dataset.  
This notebook is the **baseline** for the *reversible-LLM-training* project — subsequent notebooks will swap in reversible layers and compare peak GPU memory vs. training throughput.

| Hyperparameter | Value |
|---|---|
| `vocab_size` | 50 257 (GPT-2) |
| `n_embd` | 256 |
| `n_layer` | 8 |
| `n_head` | 8 |
| `context_len` | 256 |
| `mlp_ratio` | 4 |
| Batch size | 32 |
| Precision | bf16 (fp32 fallback) |
| Optimiser | AdamW + cosine-warmup LR |

## Cell 1 — Install Dependencies

We pin a known-good set of package versions so the notebook stays reproducible on Colab.

In [ ]:
# Install / upgrade required packages.
# Restart the Colab runtime after this cell if prompted.
import subprocess, sys

pkgs = [
    "torch>=2.2.0",
    "transformers>=4.40.0",
    "datasets>=2.19.0",
    "matplotlib>=3.8.0",
    "tqdm>=4.66.0",
]

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade"] + pkgs
)
print("All packages installed.")

## Cell 2 — Imports & Reproducibility

We fix random seeds for reproducibility and detect the available device.

In [ ]:
import math
import os
import time
import random
from dataclasses import dataclass
from typing import Optional

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import IterableDataset, DataLoader
from datasets import load_dataset
from transformers import GPT2TokenizerFast
from tqdm.auto import tqdm

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

# ── Device detection ─────────────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    DEVICE = torch.device("cpu")
    print("No GPU found — running on CPU (will be slow).")

# ── Precision: prefer bf16 on Ampere+, fall back to fp32 ─────────────────────
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float32
print(f"Precision: {'bf16' if USE_BF16 else 'fp32'}")

## Cell 3 — Model Configuration

All model and training hyper-parameters live in a single `Config` dataclass so they are easy to override for ablations.

In [ ]:
@dataclass
class Config:
    # ── Model ─────────────────────────────────────────────────────────────────
    vocab_size:   int = 50_257   # GPT-2 BPE vocabulary
    n_embd:       int = 256      # embedding / hidden dimension
    n_layer:      int = 8        # number of transformer blocks
    n_head:       int = 8        # attention heads (head_dim = n_embd // n_head = 32)
    context_len:  int = 256      # maximum sequence length
    mlp_ratio:    int = 4        # MLP hidden = mlp_ratio * n_embd
    dropout:      float = 0.0    # dropout (off for training speed)

    # ── Training ──────────────────────────────────────────────────────────────
    total_tokens:    int = 50_000_000  # 50 M training tokens
    batch_size:      int = 32          # sequences per step
    seq_len:         int = 256         # tokens per sequence (== context_len)
    learning_rate:   float = 3e-4
    weight_decay:    float = 0.1
    beta1:           float = 0.9
    beta2:           float = 0.95
    grad_clip:       float = 1.0
    warmup_steps:    int = 200
    log_interval:    int = 50    # print every N steps

    # ── Output ────────────────────────────────────────────────────────────────
    checkpoint_path: str = "baseline_checkpoint.pt"
    loss_curve_path: str = "baseline_loss_curve.png"

cfg = Config()

# Derived quantities
TOKENS_PER_STEP = cfg.batch_size * cfg.seq_len          # 32 * 256 = 8 192
TOTAL_STEPS     = cfg.total_tokens // TOKENS_PER_STEP   # ≈ 6 103
print(f"Tokens per step : {TOKENS_PER_STEP:,}")
print(f"Total steps     : {TOTAL_STEPS:,}")

## Cell 4 — Model Definition

We implement a clean, from-scratch GPT-style decoder-only transformer:

```
Input tokens
  → Token Embedding  (vocab_size × n_embd)
  + Position Embedding (context_len × n_embd)
  → n_layer × TransformerBlock
        LayerNorm → CausalSelfAttention (QKV + proj)
        LayerNorm → MLP (fc → gelu → proj)
  → Final LayerNorm
  → LM Head  (n_embd → vocab_size, weight-tied to token embedding)
```

Weight tying reduces parameters by ~12.9 M.

In [ ]:
class CausalSelfAttention(nn.Module):
    """Multi-head causal self-attention with a fused QKV projection."""

    def __init__(self, cfg: Config):
        super().__init__()
        assert cfg.n_embd % cfg.n_head == 0, "n_embd must be divisible by n_head"
        self.n_head  = cfg.n_head
        self.n_embd  = cfg.n_embd
        self.head_dim = cfg.n_embd // cfg.n_head

        # Fused QKV: one matrix of shape (n_embd, 3 * n_embd)
        self.c_attn = nn.Linear(cfg.n_embd, 3 * cfg.n_embd, bias=False)
        self.c_proj = nn.Linear(cfg.n_embd, cfg.n_embd, bias=False)
        self.attn_drop = nn.Dropout(cfg.dropout)
        self.resid_drop = nn.Dropout(cfg.dropout)

        # Causal mask (lower-triangular), registered as a non-parameter buffer
        mask = torch.tril(torch.ones(cfg.context_len, cfg.context_len))
        self.register_buffer("mask", mask.view(1, 1, cfg.context_len, cfg.context_len))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape  # batch, sequence length, embedding dim

        # Compute Q, K, V in one shot
        qkv = self.c_attn(x)                         # (B, T, 3C)
        q, k, v = qkv.split(self.n_embd, dim=-1)     # each (B, T, C)

        # Reshape to (B, n_head, T, head_dim)
        def reshape(t):
            return t.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        q, k, v = reshape(q), reshape(k), reshape(v)

        # Scaled dot-product attention
        scale = math.sqrt(self.head_dim)
        att = (q @ k.transpose(-2, -1)) / scale       # (B, n_head, T, T)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = self.attn_drop(att)

        y = att @ v                                    # (B, n_head, T, head_dim)
        y = y.transpose(1, 2).contiguous().view(B, T, C)  # re-assemble heads
        return self.resid_drop(self.c_proj(y))


class MLP(nn.Module):
    """Position-wise feed-forward network with GELU activation."""

    def __init__(self, cfg: Config):
        super().__init__()
        hidden = cfg.mlp_ratio * cfg.n_embd
        self.fc   = nn.Linear(cfg.n_embd, hidden, bias=False)
        self.proj = nn.Linear(hidden, cfg.n_embd, bias=False)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.drop(self.proj(F.gelu(self.fc(x))))


class TransformerBlock(nn.Module):
    """Pre-norm transformer block: LN → Attn + residual, LN → MLP + residual."""

    def __init__(self, cfg: Config):
        super().__init__()
        self.ln1  = nn.LayerNorm(cfg.n_embd)
        self.attn = CausalSelfAttention(cfg)
        self.ln2  = nn.LayerNorm(cfg.n_embd)
        self.mlp  = MLP(cfg)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class GPTBaseline(nn.Module):
    """GPT-style causal language model."""

    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg

        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.pos_emb = nn.Embedding(cfg.context_len, cfg.n_embd)
        self.drop    = nn.Dropout(cfg.dropout)
        self.blocks  = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layer)])
        self.ln_f    = nn.LayerNorm(cfg.n_embd)
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)

        # Weight tying: token embedding and LM head share the same weight matrix
        self.lm_head.weight = self.tok_emb.weight

        # Initialise weights
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(module: nn.Module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.LayerNorm):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)

    def forward(
        self,
        input_ids: torch.Tensor,
        targets: Optional[torch.Tensor] = None,
    ):
        B, T = input_ids.shape
        assert T <= self.cfg.context_len, "Sequence longer than context_len"

        positions = torch.arange(T, device=input_ids.device)  # (T,)
        x = self.drop(self.tok_emb(input_ids) + self.pos_emb(positions))

        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)          # (B, T, n_embd)
        logits = self.lm_head(x)  # (B, T, vocab_size)

        loss = None
        if targets is not None:
            # Flatten to (B*T, vocab_size) for cross-entropy
            loss = F.cross_entropy(
                logits.view(-1, self.cfg.vocab_size),
                targets.view(-1),
            )

        return logits, loss

print("Model classes defined.")

## Cell 5 — Instantiate Model & Count Parameters

We build the model and print a detailed parameter breakdown to confirm we are in the ~20 M range.

In [ ]:
model = GPTBaseline(cfg).to(DEVICE)

# ── Per-component parameter breakdown ────────────────────────────────────────
def count_params(module: nn.Module) -> int:
    return sum(p.numel() for p in module.parameters())

tok_emb_params = count_params(model.tok_emb)
pos_emb_params = count_params(model.pos_emb)
block_params   = [count_params(b) for b in model.blocks]
ln_f_params    = count_params(model.ln_f)
# lm_head shares weights with tok_emb → don't double-count

# Total unique parameters (weight-tied head does NOT add new params)
total_unique = sum(
    p.numel() for p in model.parameters()
)  # PyTorch deduplicates shared params? Actually no — let's do it manually.

# Manually exclude the tied weight duplication
total_unique = (
    tok_emb_params          # includes the tied lm_head weight
    + pos_emb_params
    + sum(block_params)
    + ln_f_params
)

print("=" * 55)
print(" Parameter Breakdown")
print("=" * 55)
print(f"  Token embedding (vocab_size × n_embd)   : {tok_emb_params:>12,}")
print(f"  Position embedding (ctx_len × n_embd)   : {pos_emb_params:>12,}")
for i, p in enumerate(block_params):
    print(f"  Transformer block {i:<2}                      : {p:>12,}")
print(f"  Final LayerNorm                          : {ln_f_params:>12,}")
print(f"  LM head (weight-tied — no extra params)  : {'—':>12}")
print("-" * 55)
print(f"  TOTAL unique parameters                  : {total_unique:>12,}")
print("=" * 55)

# Sanity-check expected sizes
D   = cfg.n_embd          # 256
T   = cfg.context_len     # 256
V   = cfg.vocab_size      # 50257
H   = cfg.mlp_ratio * D  # 1024
L   = cfg.n_layer         # 8

expected_tok = V * D                    # 12,865,792
expected_pos = T * D                    # 65,536
# Per block: QKV (D,3D) + proj (D,D) + MLP fc (D,H) + MLP proj (H,D) + 2 LN (2*2D)
expected_block = (D * 3*D + D * D) + (D * H + H * D) + 2 * 2 * D
expected_ln_f  = 2 * D
expected_total = expected_tok + expected_pos + L * expected_block + expected_ln_f

print(f"\n  Expected ≈ {expected_total:,} ({expected_total/1e6:.2f} M)")
print(f"  Actual    = {total_unique:,} ({total_unique/1e6:.2f} M)")
assert abs(total_unique - expected_total) < 10_000, "Unexpected parameter count!"
print("\n✓ Parameter count verified (~20 M)")

## Cell 6 — Tokenizer

We use the GPT-2 fast tokenizer from HuggingFace.  
The `eos_token` is appended between stories so the model learns story boundaries.

In [ ]:
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 has no native pad token

EOS_ID = tokenizer.eos_token_id
print(f"Vocabulary size : {tokenizer.vocab_size}")
print(f"EOS token id    : {EOS_ID}")
print(f"Max model input : {tokenizer.model_max_length}")

# Quick sanity-check
sample = "Once upon a time"
ids    = tokenizer.encode(sample)
print(f"\nSample encode: {sample!r} → {ids}")

## Cell 7 — Dataset & DataLoader

We **stream** TinyStories from HuggingFace (no full download needed).  
Tokens are concatenated into one long buffer and chunked into `seq_len`-length windows — the standard causal-LM packing approach.

```
[story_1_tokens] [EOS] [story_2_tokens] [EOS] … → flat buffer → chunks of 256
```

Each chunk `x[0..254]` is the input; `x[1..255]` is the target (next-token prediction).

In [ ]:
class TinyStoriesIterableDataset(IterableDataset):
    """
    Streams TinyStories, tokenises on the fly, and yields fixed-length
    (input_ids, targets) pairs suitable for causal language modelling.
    """

    def __init__(self, tokenizer, seq_len: int, max_tokens: int):
        self.tokenizer  = tokenizer
        self.seq_len    = seq_len
        self.max_tokens = max_tokens

    def __iter__(self):
        dataset = load_dataset(
            "roneneldan/TinyStories",
            split="train",
            streaming=True,
        )

        buffer      = []   # flat list of token ids
        total_seen  = 0

        for example in dataset:
            if total_seen >= self.max_tokens:
                break

            text = example.get("text", "")
            ids  = self.tokenizer.encode(text, add_special_tokens=False)
            buffer.extend(ids)
            buffer.append(self.tokenizer.eos_token_id)  # story boundary

            # Yield complete chunks
            while len(buffer) >= self.seq_len + 1:
                chunk = buffer[: self.seq_len + 1]
                buffer = buffer[self.seq_len + 1 :]

                x = torch.tensor(chunk[:-1], dtype=torch.long)  # (seq_len,)
                y = torch.tensor(chunk[1:],  dtype=torch.long)  # (seq_len,)
                total_seen += self.seq_len
                yield x, y

                if total_seen >= self.max_tokens:
                    break


train_dataset = TinyStoriesIterableDataset(
    tokenizer  = tokenizer,
    seq_len    = cfg.seq_len,
    max_tokens = cfg.total_tokens + cfg.batch_size * cfg.seq_len,  # small buffer
)

train_loader = DataLoader(
    train_dataset,
    batch_size  = cfg.batch_size,
    num_workers = 2,          # two background workers for tokenisation
    pin_memory  = torch.cuda.is_available(),
)

print(f"DataLoader ready — batch size {cfg.batch_size}, seq_len {cfg.seq_len}")

## Cell 8 — Optimiser & LR Schedule

We use **AdamW** with weight decay applied only to weight matrices (not embeddings or layer-norm parameters).  
The learning rate follows a **cosine decay** schedule with a linear warm-up:

$$\text{lr}(t) = \begin{cases} \frac{t}{T_{\text{warm}}} \cdot \text{lr}_{\max} & t < T_{\text{warm}} \\ \frac{1}{2}\left(1 + \cos\!\left(\pi \cdot \frac{t - T_{\text{warm}}}{T_{\text{total}} - T_{\text{warm}}}\right)\right) \cdot \text{lr}_{\max} & t \ge T_{\text{warm}} \end{cases}$$

In [ ]:
# ── Separate parameters: weight-decay vs. no-weight-decay ────────────────────
decay_params    = []
no_decay_params = []

for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    # Biases and 1-D params (LayerNorm, embeddings) → no decay
    if param.dim() == 1 or "bias" in name:
        no_decay_params.append(param)
    else:
        decay_params.append(param)

optim_groups = [
    {"params": decay_params,    "weight_decay": cfg.weight_decay},
    {"params": no_decay_params, "weight_decay": 0.0},
]

optimizer = torch.optim.AdamW(
    optim_groups,
    lr    = cfg.learning_rate,
    betas = (cfg.beta1, cfg.beta2),
    fused = torch.cuda.is_available(),  # fused kernel on CUDA
)

# ── Cosine LR schedule with warm-up ──────────────────────────────────────────
def get_lr(step: int) -> float:
    """Return the learning rate multiplier for a given step."""
    if step < cfg.warmup_steps:
        return step / max(1, cfg.warmup_steps)  # linear warm-up
    # Cosine decay from warmup_steps → total_steps
    progress = (step - cfg.warmup_steps) / max(1, TOTAL_STEPS - cfg.warmup_steps)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=get_lr)

# ── Mixed-precision scaler (no-op when using bf16 on CUDA or fp32) ───────────
# GradScaler is for fp16; bf16 doesn't need it but we keep the API consistent.
use_scaler = torch.cuda.is_available() and not USE_BF16  # only for fp16
scaler     = torch.cuda.amp.GradScaler(enabled=use_scaler)

print(f"Optimiser    : AdamW (lr={cfg.learning_rate}, wd={cfg.weight_decay})")
print(f"Schedule     : cosine with {cfg.warmup_steps}-step warm-up")
print(f"Fused kernel : {torch.cuda.is_available()}")

## Cell 9 — Training Loop

Key instrumentation:
- **GPU memory** is tracked via `torch.cuda.max_memory_allocated()`.
- **Throughput** is measured as total tokens processed divided by wall-clock seconds, **excluding the warm-up steps** (first 10 steps) to avoid JIT/compilation noise.
- Loss is logged every `log_interval` steps.

In [ ]:
# ── Reset VRAM peak counter before training ───────────────────────────────────
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

model.train()

# ── Accumulators ─────────────────────────────────────────────────────────────
loss_history     = []    # (step, loss) tuples for plotting
running_loss     = 0.0
running_count    = 0

WARMUP_SKIP_STEPS = 10   # skip first N steps when measuring throughput
tokens_for_throughput = 0
t_start: Optional[float] = None  # set after warm-up steps

loader_iter = iter(train_loader)

pbar = tqdm(range(TOTAL_STEPS), desc="Training", dynamic_ncols=True)

for step in pbar:
    # ── Fetch a batch ─────────────────────────────────────────────────────────
    try:
        x, y = next(loader_iter)
    except StopIteration:
        loader_iter = iter(train_loader)
        x, y = next(loader_iter)

    x = x.to(DEVICE, non_blocking=True)  # (B, T)
    y = y.to(DEVICE, non_blocking=True)  # (B, T)

    # ── Forward + loss ────────────────────────────────────────────────────────
    with torch.amp.autocast(device_type="cuda" if torch.cuda.is_available() else "cpu",
                             dtype=AMP_DTYPE, enabled=torch.cuda.is_available()):
        _, loss = model(x, targets=y)

    # ── Backward ──────────────────────────────────────────────────────────────
    optimizer.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
    scaler.step(optimizer)
    scaler.update()
    scheduler.step()

    # ── Throughput tracking (skip warm-up steps) ──────────────────────────────
    if step == WARMUP_SKIP_STEPS:
        t_start = time.perf_counter()
    if t_start is not None:
        tokens_for_throughput += TOKENS_PER_STEP

    # ── Logging ───────────────────────────────────────────────────────────────
    loss_val = loss.item()
    running_loss  += loss_val
    running_count += 1

    if step % cfg.log_interval == 0 or step == TOTAL_STEPS - 1:
        avg_loss = running_loss / running_count
        lr_now   = scheduler.get_last_lr()[0]
        loss_history.append((step, avg_loss))
        running_loss  = 0.0
        running_count = 0

        mem_gb = (
            torch.cuda.max_memory_allocated() / 1e9
            if torch.cuda.is_available() else 0.0
        )
        pbar.set_postfix(
            loss=f"{avg_loss:.4f}",
            lr=f"{lr_now:.2e}",
            mem=f"{mem_gb:.2f}GB",
        )

print("\nTraining complete.")

## Cell 10 — Metrics Summary

Compute and display all key metrics after training finishes.

In [ ]:
# ── Wall-clock time & throughput ──────────────────────────────────────────────
t_end = time.perf_counter()

if t_start is not None:
    training_seconds   = t_end - t_start
    tokens_per_second  = tokens_for_throughput / training_seconds
else:
    training_seconds  = float("nan")
    tokens_per_second = float("nan")

# ── GPU memory ────────────────────────────────────────────────────────────────
peak_mem_gb = (
    torch.cuda.max_memory_allocated() / 1e9
    if torch.cuda.is_available() else 0.0
)

# ── Final loss ────────────────────────────────────────────────────────────────
final_loss  = loss_history[-1][1] if loss_history else float("nan")

print("\n" + "=" * 60)
print(" TRAINING SUMMARY — BASELINE GPT".center(60))
print("=" * 60)
print(f"  Total parameters        : {total_unique:>15,}  ({total_unique/1e6:.2f} M)")
print(f"  Total tokens trained    : {cfg.total_tokens:>15,}  ({cfg.total_tokens/1e6:.0f} M)")
print(f"  Total steps             : {TOTAL_STEPS:>15,}")
print(f"  Batch size              : {cfg.batch_size:>15,}")
print(f"  Sequence length         : {cfg.seq_len:>15,}")
print(f"  Precision               : {'bf16' if USE_BF16 else 'fp32':>15}")
print("-" * 60)
print(f"  Final training loss     : {final_loss:>15.4f}")
print(f"  Peak GPU memory         : {peak_mem_gb:>14.2f} GB")
print(f"  Throughput (tok/s)      : {tokens_per_second:>15,.0f}")
print(f"  Training time (s)       : {training_seconds:>15,.1f}")
print("=" * 60)

## Cell 11 — Training Loss Curve

Plot the logged loss values and save to disk.

In [ ]:
steps, losses = zip(*loss_history) if loss_history else ([], [])

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(steps, losses, linewidth=1.5, color="steelblue", label="train loss")
ax.set_xlabel("Step")
ax.set_ylabel("Cross-entropy Loss")
ax.set_title("Baseline GPT — Training Loss on TinyStories (50 M tokens)")
ax.legend()
ax.grid(True, alpha=0.3)

# Annotate final loss
if steps:
    ax.annotate(
        f"Final: {losses[-1]:.3f}",
        xy=(steps[-1], losses[-1]),
        xytext=(-80, 20),
        textcoords="offset points",
        arrowprops=dict(arrowstyle="->", color="grey"),
        fontsize=10,
    )

plt.tight_layout()
plt.savefig(cfg.loss_curve_path, dpi=150)
print(f"Loss curve saved to: {cfg.loss_curve_path}")
plt.show()

## Cell 12 — Save Checkpoint

We save the full training state so we can resume or compare against reversible variants later.

In [ ]:
checkpoint = {
    "step":            TOTAL_STEPS,
    "model_state":     model.state_dict(),
    "optimizer_state": optimizer.state_dict(),
    "scheduler_state": scheduler.state_dict(),
    "config":          cfg.__dict__,
    "loss_history":    loss_history,
    "metrics": {
        "final_loss":        final_loss,
        "peak_mem_gb":       peak_mem_gb,
        "tokens_per_second": tokens_per_second,
        "training_seconds":  training_seconds,
        "total_params":      total_unique,
    },
}

torch.save(checkpoint, cfg.checkpoint_path)
ckpt_size_mb = os.path.getsize(cfg.checkpoint_path) / 1e6
print(f"Checkpoint saved to : {cfg.checkpoint_path}")
print(f"Checkpoint size     : {ckpt_size_mb:.1f} MB")

# Quick load verification
loaded = torch.load(cfg.checkpoint_path, map_location="cpu", weights_only=False)
assert loaded["metrics"]["total_params"] == total_unique
print("✓ Checkpoint verified (load + parameter count match).")

## Cell 13 — Qualitative Text Generation

A brief sanity-check: generate a short continuation from a prompt to verify the model has learned something reasonable.

In [ ]:
@torch.no_grad()
def generate(
    model: GPTBaseline,
    prompt: str,
    max_new_tokens: int = 80,
    temperature: float = 0.8,
    top_k: int = 40,
) -> str:
    model.eval()
    ids = tokenizer.encode(prompt, return_tensors="pt").to(DEVICE)  # (1, T)

    for _ in range(max_new_tokens):
        # Crop to context_len if needed
        ids_cond = ids[:, -cfg.context_len :]
        logits, _ = model(ids_cond)        # (1, T, V)
        logits = logits[:, -1, :] / temperature  # last token, (1, V)

        # Top-k filtering
        if top_k > 0:
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, [-1]]] = float("-inf")

        probs   = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)  # (1, 1)
        ids     = torch.cat([ids, next_id], dim=1)

        if next_id.item() == EOS_ID:
            break

    return tokenizer.decode(ids[0].tolist(), skip_special_tokens=True)


PROMPT = "Once upon a time, there was a little girl named Lily who loved"
output = generate(model, PROMPT)
print(f"Prompt : {PROMPT}")
print(f"Output : {output}")

## Summary

| Metric | Value |
|---|---|
| Model parameters | ~20 M |
| Training tokens | 50 M |
| Context length | 256 |
| Batch size | 32 |
| Precision | bf16 / fp32 fallback |
| Optimiser | AdamW + cosine-warmup |

> **Next:** `02_reversible.ipynb` — Replace `TransformerBlock` with a RevNet-style reversible block and compare peak memory & throughput against this baseline.